In [6]:
import random

# 不同约束的难度可能会影响实验结果
我们还需要测试单条指令的遵循率
因此可以设计在固定深度d=x的情况下，单独测试每条指令遵循的成功率，以及一些指令组合的遵循成功率。

In [7]:
LABELS = list('ABCDEFGHJKLMNPQRSTUVWXYZ')  # 24 个，去掉 I / O
N = len(LABELS)

CONSTRAINS = [
'''
The final line of your response MUST be exactly in this format:
Final answer: <label>
Where <label> is the single label you end at. Do not write anything after the final line.
''',
'''
The final answer label MUST be written in lowercase.
''',
'''
The response MUST contain exactly two colon characters (":").
''',
'''
The final line MUST contain exactly three whitespace-separated tokens:
"Final", "answer:", and the label.
''',
'''
Immediately before the final line, output a line containing exactly:
RESULT:
''',
'''
The first line of the response MUST be exactly:
HELLO WORLD
''',
'''
The response MUST contain the exact phrase POINTER TRACK exactly once before the final line.
''',
'''
Before the final answer, the response MUST contain the exact substring:
FROM <start> IN <d>
''',
'''
The exact phrase POINTER TRACK and the substring FROM <start> IN <d> MUST appear on the same line.
''',
'''
The response MUST contain exactly one blank line immediately before the RESULT: line.
'''
]

CONSTRAIN_PARALLEL = '''
The final line of your response MUST be exactly in this format:
Final answer: <label_1>, <label_2>, ..., <label_w>
Where there are exactly w labels, in the same order as the starting entries,
separated by a comma and one space when w > 1. Do not write anything after
the final line.
'''

def _make_constrain(k):
    '''串行约束：取前 k 条；k 超过 10 时按 10 条算。'''
    if k > len(CONSTRAINS):
        k = len(CONSTRAINS)
    return ''.join(CONSTRAINS[:k])

def _make_table(rng, n=N, min_cycle_len=2):
    '''生成指针表：每个节点一条出边、无自环；含长度 >= min_cycle_len 的环，其余节点为树。'''
    labels = LABELS[:n]
    cycle_len = rng.randint(min_cycle_len, n)
    cycle = rng.sample(labels, cycle_len)
    table = {}
    for i, node in enumerate(cycle):
        table[node] = cycle[(i + 1) % cycle_len]

    # 其余节点挂到环或已经挂好的树上，保证不会形成新环
    rest = [node for node in labels if node not in table]
    rng.shuffle(rest)
    attached = list(cycle)
    for node in rest:
        table[node] = rng.choice(attached)
        attached.append(node)

    return {node: table[node] for node in labels}

def _follow(table, start, steps):
    '''沿表走 steps 步。'''
    node = start
    for _ in range(steps):
        node = table[node]
    return node

def _table_text(table):
    '''把表排成 24 行 A -> X，便于直接拼进 prompt。'''
    return '\n'.join(f'{node} -> {table[node]}' for node in LABELS)

def make_serial_question(rng, d, k):
    '''串行指针追踪：从随机起点走 d 步，答案是一个标签。'''
    table = _make_table(rng, N, d + 1)
    start = rng.choice(LABELS)
    ans = _follow(table, start, d)
    constrain = _make_constrain(k)
    step_word = 'step' if d == 1 else 'steps'
    q = (f'The table below defines a single pointer for each entry: from an entry, the pointer tells you which entry comes next.\n\n'
         f'{_table_text(table)}\n\n'
         f'Start at {start}. Follow the pointers for exactly {d} {step_word}.\n'
         f'Question: which entry do you end at?')
    return dict(q=q, ans=ans, d=d, W=d, cons=constrain, table=table)

def make_parallel_question(rng, d, w):
    '''并行指针追踪：w 个互不相同的起点各走 d 步，答案是有序标签列表。'''
    table = _make_table(rng, N, d + 1)
    starts = rng.sample(LABELS, w)
    ans = [_follow(table, start, d) for start in starts]
    step_word = 'step' if d == 1 else 'steps'
    q = (f'The table below defines a single pointer for each entry: from an entry, the pointer tells you which entry comes next.\n\n'
         f'{_table_text(table)}\n\n'
         f'Start at these entries: {", ".join(starts)}. From each one, follow the pointers for exactly {d} {step_word}.\n'
         f'Question: what are the resulting entries, in the order given?')
    return dict(q=q, ans=ans, d=d, W=d * w, cons=CONSTRAIN_PARALLEL, table=table)

q = make_serial_question(random.Random(42), 10, 5)
print(q['q'])
print('ans:', q['ans'], 'W:', q['W'])
print(q['cons'])

qp = make_parallel_question(random.Random(42), 5, 3)
print(qp['q'])
print('ans:', qp['ans'], 'W:', qp['W'])
print(qp['cons'])


The table below defines a single pointer for each entry: from an entry, the pointer tells you which entry comes next.

A -> J
B -> T
C -> P
D -> A
E -> Z
F -> G
G -> U
H -> W
J -> H
K -> D
L -> F
M -> X
N -> P
P -> Y
Q -> B
R -> L
S -> G
T -> M
U -> V
V -> D
W -> E
X -> R
Y -> Q
Z -> C

Start at F. Follow the pointers for exactly 10 steps.
Question: which entry do you end at?
ans: Z W: 10

The final line of your response MUST be exactly in this format:
Final answer: <label>
Where <label> is the single label you end at. Do not write anything after the final line.

The final answer label MUST be written in lowercase.

The response MUST contain exactly two colon characters (":").

The final line MUST contain exactly three whitespace-separated tokens:
"Final", "answer:", and the label.

Immediately before the final line, output a line containing exactly:
RESULT:

The table below defines a single pointer for each entry: from an entry, the pointer tells you which entry comes next.

A -> J
B 

# Generate the whole dataset

In [8]:
random_seed_generator = random.Random(42)
max_d = 20
max_k = 10
q_in_each_grid = 20
q_matrix = []

for d in range(1, max_d + 1):
    k_list = []
    for k in range(1, max_k + 1):
        q_list = []
        for i in range(q_in_each_grid):
            random_seed = random_seed_generator.randint(0, 2**32)
            q = make_serial_question(random.Random(random_seed), d, k)
            q_list.append(q)
        k_list.append(q_list)
    q_matrix.append(k_list)

# q_matrix[depth][constrain_num][question_number]
print(q_matrix[1][9][1])
print('serial rows:', sum(len(q_list) for k_list in q_matrix for q_list in k_list))


{'q': 'The table below defines a single pointer for each entry: from an entry, the pointer tells you which entry comes next.\n\nA -> M\nB -> L\nC -> V\nD -> S\nE -> R\nF -> D\nG -> B\nH -> Y\nJ -> E\nK -> W\nL -> F\nM -> V\nN -> Q\nP -> R\nQ -> J\nR -> T\nS -> W\nT -> G\nU -> F\nV -> Y\nW -> M\nX -> N\nY -> E\nZ -> T\n\nStart at L. Follow the pointers for exactly 2 steps.\nQuestion: which entry do you end at?', 'ans': 'D', 'd': 2, 'W': 2, 'cons': '\nThe final line of your response MUST be exactly in this format:\nFinal answer: <label>\nWhere <label> is the single label you end at. Do not write anything after the final line.\n\nThe final answer label MUST be written in lowercase.\n\nThe response MUST contain exactly two colon characters (":").\n\nThe final line MUST contain exactly three whitespace-separated tokens:\n"Final", "answer:", and the label.\n\nImmediately before the final line, output a line containing exactly:\nRESULT:\n\nThe first line of the response MUST be exactly:\nHELL

# Generate the parallel dataset

`d × w`：`w` 个起点各走 `d` 步，`W = d × w`；每格 10 题。


In [9]:
random_seed_generator_parallel = random.Random(43)  # 与串行分开，避免意外共用同一批表
max_w = 20
parallel_q_in_each_grid = 10
w_matrix = []

for d in range(1, max_d + 1):
    w_list = []
    for w in range(1, max_w + 1):
        q_list = []
        for i in range(parallel_q_in_each_grid):
            random_seed = random_seed_generator_parallel.randint(0, 2**32)
            q = make_parallel_question(random.Random(random_seed), d, w)
            q_list.append(q)
        w_list.append(q_list)
    w_matrix.append(w_list)

# w_matrix[depth][width][question_number]
print(w_matrix[1][9][1])
print('parallel rows:', sum(len(q_list) for w_list in w_matrix for q_list in w_list))


{'q': 'The table below defines a single pointer for each entry: from an entry, the pointer tells you which entry comes next.\n\nA -> J\nB -> A\nC -> B\nD -> W\nE -> K\nF -> M\nG -> M\nH -> S\nJ -> F\nK -> H\nL -> G\nM -> P\nN -> J\nP -> H\nQ -> E\nR -> W\nS -> T\nT -> A\nU -> W\nV -> H\nW -> T\nX -> M\nY -> D\nZ -> S\n\nStart at these entries: X, N, U, H, A, L, Y, D, T, R. From each one, follow the pointers for exactly 2 steps.\nQuestion: what are the resulting entries, in the order given?', 'ans': ['P', 'F', 'T', 'T', 'F', 'M', 'W', 'T', 'J', 'T'], 'd': 2, 'W': 20, 'cons': '\nThe final line of your response MUST be exactly in this format:\nFinal answer: <label_1>, <label_2>, ..., <label_w>\nWhere there are exactly w labels, in the same order as the starting entries,\nseparated by a comma and one space when w > 1. Do not write anything after\nthe final line.\n', 'table': {'A': 'J', 'B': 'A', 'C': 'B', 'D': 'W', 'E': 'K', 'F': 'M', 'G': 'M', 'H': 'S', 'J': 'F', 'K': 'H', 'L': 'G', 'M': 

In [10]:
import json
from datasets import load_dataset

rows = []
for d, k_list in enumerate(q_matrix):
    for k, q_list in enumerate(k_list):
        for i, q in enumerate(q_list):
            rows.append({
                'd': d + 1,
                'k': k + 1,
                'i': i + 1,
                'q': q['q'],
                'ans': q['ans'],
                'W': q['W'],
                'cons': q['cons'],
                'table': q['table']
            })

save_path = 'synthesis_q_k_dataset.jsonl'
with open(save_path, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

rows_parallel = []
for d, w_list in enumerate(w_matrix):
    for w, q_list in enumerate(w_list):
        for i, q in enumerate(q_list):
            rows_parallel.append({
                'd': d + 1,
                'w': w + 1,
                'i': i + 1,
                'q': q['q'],
                'ans': q['ans'],
                'W': q['W'],
                'cons': q['cons'],
                'table': q['table']
            })

parallel_save_path = 'synthesis_d_w_dataset.jsonl'
with open(parallel_save_path, 'w', encoding='utf-8') as f:
    for r in rows_parallel:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

ds_serial = load_dataset('json', data_files=save_path, split='train')
ds_parallel = load_dataset('json', data_files=parallel_save_path, split='train')
print(ds_serial)
print(ds_parallel)


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['d', 'k', 'i', 'q', 'ans', 'W', 'cons', 'table'],
    num_rows: 4000
})
Dataset({
    features: ['d', 'w', 'i', 'q', 'ans', 'W', 'cons', 'table'],
    num_rows: 4000
})
